# Hyperparameter Tunnig

This script is using the data pipeline to clean the data.
It will use Hyperopt for hyperparameter tuning and safe the best model via mlflow.

The models will be tested against:
- 1 day
- 1 week
- 2 weeks
- 4 weeks
- 1 quarter
- 2 quarters
- 3 quarters
- 4 quarters

As well as based on data need, this will be evaluated based on CV.

The models to be tuned are:
- SARIMAX
- Tripple Exponential Smoothing
- Prophet
- XG Boost
- Linear Regression
- Random Forest
- LSTM
- Temporal Fusion Transformer (TFT)
- Deep Autoregression Models

# Libraries

In [26]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import sys
import os
from darts import TimeSeries
from darts.models import Prophet, ARIMA, ExponentialSmoothing
from darts.metrics import mae, mape, rmse
from hyperopt import hp

# Add the project root to the python path
sys.path.append(os.path.abspath(".."))
from src.processing import DateFeatureTransformer, TimeSeriesWrangler, LagFeatureTransformer, WindowFeatureTransformer
from src.evaluation import DartsObjective, TimeSeriesOptimizer


# Loading Data

In [27]:
# define path
path = "../data/raw/"

In [28]:
# oil data
oil_df = pd.read_csv(path + "oil.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='dcoilwtico', 
    freq='D', 
    fill_method='ffill'
)

# Run the cleaning logic
oil = wrangler.clean(oil_df)

In [29]:
# timeseries data
timeseries_df = pd.read_csv(path + "timeseries.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='unit_sales', 
    freq='D', 
    fill_method='zeros'
)

# Run the cleaning logic
timeseries = wrangler.clean(timeseries_df)

/Users/moe/Developer/work-projects/MIST/TimeSeries_April2026/corporacion_favorita_grocery_sales_forecasting/src/processing/wrangler.py:41: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df = resampler[self.fill_col].fillna(0)


# Variables

In [30]:
# Defining constants
random_seed = 42
# Change these if you df has different column names
target_col = 'unit_sales'
time_col = 'date'

# SARIMAX

In [31]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['year', 'month', 'day_of_week', 'is_weekend', 'is_holiday', 'is_payday'], payday_val=15, country='EC', drop_date_col=False))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')
# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()
series = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=target_col, freq='D')
exog = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=all_exog_features, freq='D')

# Define your models and search spaces
registry = {
    'SARIMAX': {
        'class': ARIMA,
        'space': {
            # 1. Feature Selection: Toggles each feature on or off
            'selected_features': [hp.choice(f'feat_{f}', [None, f]) for f in all_exog_features],
            
            # 2. Standard SARIMAX Params
            'p': hp.quniform('p', 0, 3, 1),
            'd': hp.choice('d', [0, 1]),
            'q': hp.quniform('q', 0, 3, 1),
            
            # 3. Seasonal Params (Weekly Seasonality for Ecuador Sales)
            'seasonal_order': (
            hp.quniform('P', 0, 2, 1),
            hp.choice('D', [0, 1]),
            hp.quniform('Q', 0, 2, 1),
            7),
            'trend': hp.choice('trend', ['n', 'c', 't', 'ct'])
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name="SARIMAX_7")

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=series,                  # Your Darts TimeSeries
        horizon=7,                      # 7-day forecast
        metric=mape,                    # Optimization metric
        exog=exog,                      # Your Darts exogenous TimeSeries
        max_evals=50                    # Run 50 trials per model
    )



2026/04/29 19:41:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/29 19:41:13 INFO mlflow.store.db.utils: Updating database tables
2026/04/29 19:41:13 INFO mlflow.tracking.fluent: Experiment with name 'SARIMAX_7' does not exist. Creating a new experiment.


Running Hyperopt for SARIMAX up to 50 evals...
  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

IndexError: The type of your index was not matched.

job exception: The type of your index was not matched.



  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]


IndexError: The type of your index was not matched.

# Tripple Exponential Smooting

# Prophet

# XG Boost

# Linear Regression

# Random Forest

# LSTM

# TFT

# Deep Autoregression Models